# Chi-square residuals

Inspect dataset checks, clr_cc outliers, and standardized residuals for the lc_type × clr contingency table.

## Setup

In [10]:
import polars as pl
from scipy.stats import chi2_contingency
import numpy as np

## Dataset checks

In [11]:
df = pl.read_csv("../../../dataset/Capstone2025_nsi_lvl9_with_landcover_and_color.csv.gz")
df

h3,fips,st_damcat,bldgtype,lc_type,loc,clr,clr_cc
str,i64,str,str,str,str,str,i64
"""8929a960d6fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.828147398273 37.072…","""orange""",8
"""8929a978c07ffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.63428156983 36.9838…","""orange""",8
"""892836da147ffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.464731172229 37.323…","""orange""",11
"""892836c644fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.787252301526 37.429…","""orange""",12
"""892836d736fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.716239893409 37.339…","""orange""",1
…,…,…,…,…,…,…,…
"""892836da15bffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.457547536701 37.320…","""olive""",6
"""892836c226bffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.591661571591 37.401…","""olive""",1
"""892836d121bffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.492292055735 37.305…","""olive""",1


In [12]:
df.filter(pl.col("loc") == df[0, "loc"])

h3,fips,st_damcat,bldgtype,lc_type,loc,clr,clr_cc
str,i64,str,str,str,str,str,i64
"""8929a960d6fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.828147398273 37.072…","""orange""",8
"""8929a960d6fffff""",6047,"""RES""","""W""","""urban+crop""","""POINT(-120.828147398273 37.072…","""coffee""",20
"""8929a960d6fffff""",6047,"""RES""","""W""","""urban+crop""","""POINT(-120.828147398273 37.072…","""crimson""",5
"""8929a960d6fffff""",6047,"""RES""","""W""","""urban+crop""","""POINT(-120.828147398273 37.072…","""gold""",1
"""8929a960d6fffff""",6047,"""RES""","""W""","""urban+crop""","""POINT(-120.828147398273 37.072…","""gray""",6
…,…,…,…,…,…,…,…
"""8929a960d6fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.828147398273 37.072…","""gold""",1
"""8929a960d6fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.828147398273 37.072…","""gray""",4
"""8929a960d6fffff""",6047,"""RES""","""M""","""urban+crop""","""POINT(-120.828147398273 37.072…","""lavender""",7


In [13]:
df.group_by("clr").agg(pl.col("clr_cc").sum().alias("n_buildings")).sort("n_buildings", descending=True)


clr,n_buildings
str,i64
"""cocoa""",2030453
"""orange""",722041
"""brown""",712253
"""terracotta""",700726
"""alabaster""",624416
…,…
"""auburn""",4995
"""tan""",4776
"""plum""",3600


In [14]:
df.group_by("st_damcat").agg(pl.col("clr_cc").sum().alias("n_buildings")).sort("n_buildings", descending=True)


st_damcat,n_buildings
str,i64
"""RES""",9372135
"""COM""",286950
"""IND""",46525
"""PUB""",22629


In [15]:
df.group_by("bldgtype").agg(pl.col("clr_cc").sum().alias("n_buildings")).sort("n_buildings", descending=True)


bldgtype,n_buildings
str,i64
"""W""",4984123
"""M""",4189954
"""H""",282245
"""C""",183064
"""S""",88853


In [16]:
df["clr_cc"].describe()

statistic,value
str,f64
"""count""",2.417766e6
"""null_count""",0.0
"""mean""",4.023648
"""std""",4.631857
"""min""",0.0
"""25%""",1.0
"""50%""",2.0
"""75%""",5.0
"""max""",377.0


In [17]:
lc_clr = df.group_by(["lc_type", "clr"]).agg(pl.col("clr_cc").sum().alias("n_buildings"))
lc_clr = (
    lc_clr
    .with_columns((pl.col("n_buildings") / pl.col("n_buildings").sum().over("lc_type")).alias("pct"))
    .sort(["lc_type", "n_buildings"], descending=[False, True])
)
lc_clr


lc_type,clr,n_buildings,pct
str,str,i64,f64
"""barren""","""cocoa""",9809,0.41369
"""barren""","""orange""",2899,0.122264
"""barren""","""alabaster""",2425,0.102273
"""barren""","""navy""",2418,0.101978
"""barren""","""red""",1643,0.069293
…,…,…,…
"""urban+shrub""","""plum""",50,0.000174
"""urban+shrub""","""bar""",40,0.000139
"""urban+shrub""","""auburn""",40,0.000139


In [18]:
fips_clr = df.group_by(["fips", "clr"]).agg(pl.col("clr_cc").sum().alias("n_buildings"))
fips_clr = (
    fips_clr
    .with_columns((pl.col("n_buildings") / pl.col("n_buildings").sum().over("fips")).alias("pct"))
    .sort(["fips", "n_buildings"], descending=[False, True])
)
fips_clr


fips,clr,n_buildings,pct
i64,str,i64,f64
6001,"""cocoa""",112563,0.286199
6001,"""orange""",61310,0.155885
6001,"""red""",42728,0.108639
6001,"""navy""",40340,0.102567
6001,"""purple""",38303,0.097388
…,…,…,…
6115,"""blue""",346,0.017479
6115,"""beige""",342,0.017277
6115,"""yellow""",321,0.016216


## clr_cc outliers (IQR)

In [19]:
q1 = df["clr_cc"].quantile(0.25)
q3 = df["clr_cc"].quantile(0.75)
iqr = q3 - q1
upper_bound = q3 + 1.5 * iqr

f"Q1: {q1}, Q3: {q3}, IQR: {iqr}, Upper bound: {upper_bound}"

'Q1: 1.0, Q3: 5.0, IQR: 4.0, Upper bound: 11.0'

In [20]:
clr_cc_outliers = df.filter(pl.col("clr_cc") > upper_bound)
f"Outliers: {len(clr_cc_outliers):,} rows ({100 * len(clr_cc_outliers) / len(df):.2f}%)"

'Outliers: 140,828 rows (5.82%)'

In [21]:
clr_cc_outliers["clr_cc"].describe()

statistic,value
str,f64
"""count""",140828.0
"""null_count""",0.0
"""mean""",17.794998
"""std""",8.015995
"""min""",12.0
"""25%""",13.0
"""50%""",15.0
"""75%""",20.0
"""max""",377.0


### Findings: clr_cc outliers

- **Upper bound**: 11 (Q3 + 1.5×IQR)
- **Outliers**: 140,828 rows
- Outlier distribution: median=15, mean=17.8, max=377

Some H3 cells have unusually high color counts. These could represent dense urban areas or data outliers?

## Chi-square residuals for lc_type and clr

In [22]:
contingency = (
    df.group_by(["lc_type", "clr"])
    .agg(pl.col("clr_cc").sum().alias("n_buildings"))
    .pivot(on="clr", index="lc_type", values="n_buildings")
    .fill_null(0)
)
lc_types = contingency["lc_type"].to_list()
clr_cols = [c for c in contingency.columns if c != "lc_type"]
observed = contingency.select(clr_cols).to_numpy()
chi2, p, dof, expected = chi2_contingency(observed)
residuals = (observed - expected) / np.sqrt(expected)
residuals_df = pl.DataFrame({"lc_type": lc_types})
for i, col in enumerate(clr_cols):
    residuals_df = residuals_df.with_columns(pl.Series(name=col, values=residuals[:, i]))

f"Chi-square: {chi2:,.0f}, p-value: {p:.2e}, dof: {dof}"


'Chi-square: 2,867,941, p-value: 0.00e+00, dof: 444'

In [23]:
residuals_df

lc_type,gold,red,beige,foo,plum,ivory,green,purple,crimson,emerald,lemon,sage,lavender,aqua,lilac,olive,yellow,gray,cocoa,orange,blue,amber,bar,verde,aquamarine,grey,maroon,terracotta,alabaster,azure,brown,navy,scarlet,auburn,coffee,indigo,sienna,tan
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""shrub""",-11.845892,-25.90135,52.85092,-45.398169,-2.785478,-25.391187,-2.744393,-54.433361,-7.039473,-1.666162,-50.816741,36.132162,-11.921263,49.402466,-41.319519,-26.338434,-7.920908,30.961325,37.195936,-30.467117,24.152595,-44.413022,9.340925,13.225319,16.341988,0.2044,0.494002,-49.93626,52.948425,30.455765,76.364403,35.260488,-14.647406,-8.297592,32.621982,15.982113,-4.114717,59.057069
"""barren""",-4.749495,7.783591,4.351922,-18.470524,-0.936617,-22.608979,8.480724,-14.073014,-4.581257,0.148115,-26.075955,-20.173623,-24.380232,4.040004,-23.06762,-23.291388,-2.313159,17.527338,69.086154,27.154344,-11.891291,-5.808288,-3.057984,-8.134123,38.903336,-3.993121,-0.873636,-39.028065,23.149149,-16.261914,-37.969226,51.464034,-6.216232,-2.916,-19.104496,-6.314172,-6.685836,-3.41185
"""urban+grass""",-7.873298,42.301508,-60.883512,-31.602846,-7.92918,-116.047576,-66.471064,-1.492646,-11.247472,-1.279238,-91.664548,152.636675,224.431311,58.552419,-73.107412,-125.308628,67.563089,30.488972,-196.710855,-51.230457,69.905908,86.117774,0.649339,-33.090299,-8.326298,-51.634785,3.686851,17.248942,121.873935,55.456154,287.299215,-22.493049,-37.527436,-20.713726,-93.155553,73.379699,21.264627,22.291426
"""urban+barren""",-16.617015,189.56302,-14.352421,-71.545231,-5.739529,-76.680303,1.843464,77.518669,-14.627801,-5.0362,-63.242385,-74.407365,-85.458126,-3.149095,-63.252029,-99.817465,-10.156052,-20.408313,35.511788,190.174216,-37.059403,136.892728,-12.318229,-28.103306,-5.297119,-22.046479,-3.574704,-123.558807,45.405718,-61.581656,-137.687382,149.040531,-20.825155,-11.391522,-74.432815,-23.636653,-26.307912,-11.787906
"""forest""",-23.472815,-138.271577,-2.400898,-102.07045,-11.544236,-100.797286,160.261007,-106.572108,-20.737773,37.746611,-124.712052,191.979095,-10.96574,47.110924,-97.507986,181.801688,-19.067819,85.050322,-116.350429,-100.642166,4.771132,-123.122813,-0.01789,262.008704,37.714949,62.615096,-6.917005,-101.640064,38.71668,-49.015719,333.397201,-71.736809,-28.067671,-15.731084,354.663645,132.873087,-36.040644,116.643518
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""grass""",-4.358976,-15.743941,2.459428,-17.157687,0.862673,10.648679,-6.118692,-15.605725,-3.491787,-1.36131,-20.582726,-3.251146,-10.650875,3.641086,-17.426182,-11.612936,-3.455281,22.743177,6.569995,-1.863471,-7.903615,-13.257817,16.277118,-1.726346,0.924765,21.603093,-1.253431,-20.713302,87.842103,5.44626,-5.736948,14.203332,-5.099025,-2.862104,-3.451755,3.733032,-6.582039,6.134195
"""urban""",6.62488,50.692343,47.311799,98.854667,-4.884406,209.289662,-120.169655,75.555124,-69.074552,-24.186178,187.482147,-210.745326,-189.587457,-71.91649,135.994017,-14.483859,-59.206917,-118.80651,210.340129,86.229722,-12.86967,16.39811,-32.997077,-125.800876,-68.765561,5.64482,2.052033,48.266051,-99.305241,19.751825,-442.47014,72.982633,51.417864,31.872818,-179.162208,-136.803853,-17.368728,-52.76706
"""urban+shrub""",-13.403862,21.871648,-4.957349,7.136199,-5.454064,-0.539182,-38.313561,40.501409,-8.786378,-5.081091,32.558597,-44.823334,-38.634066,-0.066255,57.070514,-3.295691,-3.191786,-10.607403,17.949096,34.992183,-22.443388,34.253248,-9.382394,-21.303054,-10.254228,-24.205465,1.455302,25.636211,-1.559613,-24.369042,-81.071765,17.754619,-18.390349,-8.844278,-52.16175,-21.642325,-8.892067,-2.603381


### Combinations rarer than expected under independence

In [24]:
residuals_long = residuals_df.unpivot(index="lc_type", variable_name="clr", value_name="std_residual")
residuals_long.sort("std_residual").head(10)

lc_type,clr,std_residual
str,str,f64
"""urban""","""brown""",-442.47014
"""urban+crop""","""cocoa""",-212.042237
"""urban""","""sage""",-210.745326
"""urban+grass""","""cocoa""",-196.710855
"""urban""","""lavender""",-189.587457
"""urban""","""coffee""",-179.162208
"""urban+forest""","""red""",-147.087713
"""urban+forest""","""navy""",-144.659511
"""urban+forest""","""ivory""",-139.450435


### Combinations more common than expected under independence

In [25]:
residuals_long.sort("std_residual", descending=True).head(10)

lc_type,clr,std_residual
str,str,f64
"""urban+forest""","""brown""",426.495498
"""forest""","""coffee""",354.663645
"""forest""","""brown""",333.397201
"""urban+grass""","""brown""",287.299215
"""urban+forest""","""green""",270.354234
"""forest""","""verde""",262.008704
"""urban+crop""","""lavender""",235.985436
"""urban+grass""","""lavender""",224.431311
"""urban""","""cocoa""",210.340129


## Interpretation and handoff

Chi-square test: p ≈ 0 — color and land cover are strongly dependent (not independent).

Forest areas associated with coffee, brown, verde, olive, sage and not really red, amber, lemon. Urban areas avoid earth tones are more likely to be lemon. So synthetic color assignment is related to land cover type.